# Unidad 4 · Cuaderno 04 · Comunicación de resultados

**Modelación y Simulación Computacional** · Maestría en Ingeniería, Universidad de Sucre, periodo 2026-2

**Unidad 4.** Validación, interpretación y comunicación de resultados
· **Subtema del plan 4.4**

Este cuaderno ejecuta lo que el libro expone en la sección 4.6 y la subsección 4.5.1. El texto no
repite la teoría, remite a ella por número de definición, de teorema, de
ejemplo, de listado, de tabla o de figura, y se ocupa de reproducir los
resultados publicados y de verificarlos.

**Autor.** Prof. Daniel Otero Meza, Ing., Ph.D.

<!-- ENLACE_COLAB -->
[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/msc-unisucre/msc2026-material/blob/main/03_cuadernos/Unidad4/U4_04_comunicacion_de_resultados.ipynb)

## Objetivos de aprendizaje

1. Trasladar las secciones del informe técnico a las del artículo científico con la correspondencia de la Figura 4.12 del libro.
2. Construir la plantilla de un informe reproducible que cumpla la Definición 4.11, con datos crudos de solo lectura y semillas fijas.
3. Ejecutar la lista de verificación de la Tabla 4.6 sobre un repositorio y obtener el veredicto punto por punto.
4. Llenar la ficha de métodos de la Tabla 4.7 con el entorno, las versiones y las semillas leídas del propio sistema.
5. Declarar el uso de asistentes de programación con el alcance y la verificación aplicada, según pide la sección 4.6.1.

## Puesta a punto

In [ ]:
# Puesta a punto. Detecta el entorno e instala solo lo que falte.
import importlib
import subprocess
import sys

EN_COLAB = "google.colab" in sys.modules


def asegurar(paquetes: dict[str, str]) -> None:
    """Instala los paquetes cuyo módulo no se encuentre en el entorno."""
    faltantes = [p for p, m in paquetes.items()
                 if importlib.util.find_spec(m) is None]
    if faltantes:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *faltantes],
                       check=True)


asegurar({"numpy": "numpy", "scipy": "scipy", "pandas": "pandas",
          "matplotlib": "matplotlib", "sympy": "sympy"})
print("Entorno listo. Colab:", EN_COLAB)

In [ ]:
%matplotlib inline
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

SEMILLA = 20262
rng = np.random.default_rng(SEMILLA)

PALETA = {"azul": "#1F4E79", "rojo": "#B3251E", "verde": "#2E7D32",
           "naranja": "#E07B00", "gris": "#5A5A5A", "morado": "#6A3D9A"}

plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 200, "savefig.bbox": "tight",
    "font.size": 10, "axes.grid": True, "grid.alpha": 0.25,
    "axes.prop_cycle": plt.cycler(color=list(PALETA.values())),
})

# Bandera de los ejercicios guiados. En la versión de trabajo vale False
# para que el cuaderno corra completo aunque falten celdas por resolver.
REVISAR = False


def verificar(nombre: str, obtenido, esperado: float,
              tol: float = 1.0e-3) -> bool:
    """Compara un resultado con el valor esperado sin detener el cuaderno."""
    if obtenido is None or (isinstance(obtenido, float) and np.isnan(obtenido)):
        print(f"[pendiente] {nombre}, la celda marcada COMPLETE sigue sin resolver")
        return False
    escala = abs(esperado) if esperado != 0.0 else 1.0
    error = abs(float(obtenido) - esperado) / escala
    estado = "ok" if error <= tol else "revisar"
    print(f"[{estado}] {nombre}, obtenido {float(obtenido):.6g}, "
          f"esperado {esperado:.6g}, error relativo {error:.2e}")
    if REVISAR:
        assert error <= tol, f"{nombre} no coincide con el valor esperado"
    return error <= tol


print("Semilla del curso:", SEMILLA)

In [ ]:
# Acceso a datos/ que funciona en Colab y en local, sin rutas absolutas.
# Si la carpeta no viaja con el cuaderno, las series se reconstruyen con la
# semilla del curso y con las cifras que el libro publica.

def carpeta_datos() -> Path:
    """Ubica datos/ subiendo por el árbol, o la crea junto al cuaderno."""
    base = Path.cwd()
    for nivel in [base, *base.parents][:4]:
        for candidata in (nivel / "datos", nivel / "03_cuadernos" / "datos"):
            if candidata.is_dir():
                return candidata
    destino = base / "datos"
    destino.mkdir(parents=True, exist_ok=True)
    return destino


def _cinetica_monod() -> pd.DataFrame:
    return pd.DataFrame({
        "S_g_L": [0.5, 1.0, 2.0, 3.0, 5.0, 8.0, 12.0, 18.0, 25.0, 35.0],
        "mu_1_h": [0.0702, 0.1248, 0.1919, 0.2496, 0.2761,
                   0.3205, 0.3689, 0.3709, 0.3770, 0.3773]})


def _secado_calibracion() -> pd.DataFrame:
    t = np.array([0.25, 0.5, 0.75, 1.0, 1.5, 2.0, 2.5, 3.0,
                  4.0, 5.0, 6.0, 7.0, 8.0])
    gen = np.random.default_rng(SEMILLA)
    mr = np.round(np.exp(-0.350 * t**1.15) + gen.normal(0.0, 0.008, t.size), 4)
    return pd.DataFrame({"t_h": t, "MR": mr})


def _secado_validacion() -> pd.DataFrame:
    t = np.array([0.5, 1.0, 1.5, 2.0, 2.75, 3.5, 4.5, 5.5, 6.5, 8.0])
    gen = np.random.default_rng(SEMILLA + 1)
    mr = np.round(np.exp(-0.362 * t**1.15) + gen.normal(0.0, 0.008, t.size), 4)
    return pd.DataFrame({"t_h": t, "MR": mr})


def _caudal_mensual() -> pd.DataFrame:
    obs = [6.21, 5.01, 4.30, 9.39, 14.85, 18.28, 16.85, 17.12, 21.51, 22.75,
           19.33, 12.13, 7.65, 6.14, 4.54, 8.58, 14.73, 21.82, 14.36, 17.57,
           24.54, 31.26, 21.30, 11.27, 9.71, 6.09, 5.39, 10.86, 23.94, 20.68,
           19.39, 22.16, 32.76, 38.82, 21.33, 12.99, 5.97, 5.83, 5.07, 8.20,
           18.23, 18.93, 12.72, 16.63, 22.38, 27.76, 18.00, 12.27]
    sim = [7.70, 4.70, 4.43, 7.83, 14.55, 16.00, 15.33, 17.97, 20.37, 24.45,
           17.19, 13.27, 10.40, 6.93, 5.40, 10.19, 16.72, 22.35, 13.78, 17.44,
           24.20, 29.67, 17.56, 13.20, 12.94, 2.59, 5.60, 12.74, 21.94, 16.05,
           18.74, 17.99, 27.76, 24.48, 15.41, 12.64, 6.83, 6.11, 5.24, 11.99,
           19.88, 20.51, 12.28, 17.89, 19.38, 19.28, 14.48, 8.00]
    return pd.DataFrame({"mes": np.arange(1, 49),
                         "periodo": ["calibracion"] * 24 + ["validacion"] * 24,
                         "Q_obs_m3_s": obs, "Q_sim_m3_s": sim})


def _arreglo_fotovoltaico() -> pd.DataFrame:
    return pd.DataFrame({"configuracion": ["Base", "Optima", "Sombreado"],
                         "H_kWh_m2": [1980.0, 2035.0, 1910.0],
                         "u_rel_H": [0.04, 0.04, 0.04]})


def _entradas_vertedero() -> pd.DataFrame:
    return pd.DataFrame({"magnitud": ["C_d", "b", "h"],
                         "unidad": ["1", "m", "m"],
                         "valor": [0.620, 0.500, 0.150],
                         "u_tipica": [0.015, 0.0010, 0.0015]})


CONSTRUCTORES = {
    "cinetica_monod.csv": _cinetica_monod,
    "secado_maiz_calibracion.csv": _secado_calibracion,
    "secado_maiz_validacion.csv": _secado_validacion,
    "caudal_mensual.csv": _caudal_mensual,
    "arreglo_fotovoltaico.csv": _arreglo_fotovoltaico,
    "entradas_vertedero.csv": _entradas_vertedero,
}

CARPETA_DATOS = carpeta_datos()


def leer_datos(nombre: str) -> pd.DataFrame:
    """Lee un archivo de datos/ y lo reconstruye si no está presente."""
    ruta = CARPETA_DATOS / nombre
    if not ruta.exists():
        CONSTRUCTORES[nombre]().to_csv(ruta, index=False)
    return pd.read_csv(ruta)


print("Carpeta de datos:", CARPETA_DATOS.name)

## 1. Del informe técnico al artículo científico

El paso del informe al artículo no es un cambio de formato sino de
destinatario. El informe responde a un cliente que necesita decidir
algo concreto, mientras que el artículo se dirige a una comunidad que
necesita saber algo general, y por eso exige una afirmación
defendible que nadie haya establecido antes. La Figura 4.12 del libro
muestra la correspondencia entre ambas estructuras.

La regla de organización que mejor resultado da consiste en tratar
cada sección como respuesta a una pregunta única y en subordinar el
escrito a una sola afirmación central. La introducción establece qué
se sabe, qué falta y qué aporta el trabajo; los métodos entregan lo
necesario para repetirlo; los resultados presentan evidencia sin
interpretarla; y la discusión interpreta y declara los límites.

In [ ]:
CORRESPONDENCIA = pd.DataFrame([
    {"informe": "Problema y alcance", "articulo": "Introducción",
     "pregunta": "qué se decide y por qué importa"},
    {"informe": "Modelo y supuestos", "articulo": "Métodos",
     "pregunta": "qué se representó y qué se dejó fuera"},
    {"informe": "Verificación", "articulo": "Métodos",
     "pregunta": "el código resuelve las ecuaciones escritas"},
    {"informe": "Calibración", "articulo": "Métodos",
     "pregunta": "de dónde salen los parámetros y con qué precisión"},
    {"informe": "Validación", "articulo": "Resultados",
     "pregunta": "qué tan bien predice lo que no vio"},
    {"informe": "Escenarios", "articulo": "Resultados",
     "pregunta": "qué pasa si cambian las condiciones"},
    {"informe": "Incertidumbre", "articulo": "Resultados",
     "pregunta": "cuánto vale el margen y de dónde viene"},
    {"informe": "Conclusiones", "articulo": "Discusión",
     "pregunta": "qué se afirma y con qué respaldo"},
    {"informe": "Limitaciones", "articulo": "Discusión",
     "pregunta": "dónde deja de valer"},
    {"informe": "Anexo reproducible", "articulo": "Disponibilidad",
     "pregunta": "cómo lo repite otro grupo"},
])
print(CORRESPONDENCIA.to_string(index=False))
print(f"\nEl informe técnico tiene {len(CORRESPONDENCIA)} secciones,")
print("las diez que la sección 4.5.1 del libro enumera.")
assert len(CORRESPONDENCIA) == 10
assert set(CORRESPONDENCIA["articulo"]) == {
    "Introducción", "Métodos", "Resultados", "Discusión",
    "Disponibilidad"}

## 2. La plantilla del informe reproducible

La Definición 4.11 del libro exige que las figuras, las tablas y las
cifras se regeneren íntegramente a partir de los datos crudos
ejecutando el código que las acompaña, en un entorno declarado y con
las semillas fijadas, sin que ninguna cifra se transcriba a mano. La
Figura 4.11 muestra la correspondencia con los artefactos del
repositorio.

La celda siguiente crea esa plantilla dentro de una carpeta temporal,
de modo que el cuaderno no ensucie el proyecto del estudiante. La
misma estructura sirve para el mini proyecto de la asignatura.

In [ ]:
import tempfile

RAIZ_DEMO = Path(tempfile.mkdtemp(prefix="informe_u4_"))

ARBOL = [
    "datos/crudos", "datos/procesados", "codigo", "figuras",
    "tablas", "informe",
]
for carpeta in ARBOL:
    (RAIZ_DEMO / carpeta).mkdir(parents=True, exist_ok=True)

print("Estructura creada en una carpeta temporal:")
for carpeta in ARBOL:
    print(f"  {carpeta}/")

### 2.1 Los archivos que la plantilla necesita

Cada archivo responde a una condición de la Definición 4.11. El
entorno declarado permite reconstruir las versiones, la semilla fija
hace repetible el muestreo, la licencia permite reutilizar y el guion
de reconstrucción regenera todo con un solo comando.

In [ ]:
import sys

PLANTILLA_INFORME = [
    "# Informe de simulación",
    "",
    "Plantilla del informe reproducible de la Unidad 4, con las diez",
    "secciones que la sección 4.5.1 del libro enumera. Cada cifra de",
    "este documento proviene de codigo/reconstruir.py y ninguna se",
    "transcribe a mano.",
    "",
    "## 1. Problema y alcance",
    "",
    "Enuncie la pregunta de ingeniería y la decisión que apoya.",
    "",
    "## 2. Modelo y supuestos",
    "",
    "Ecuaciones gobernantes, hipótesis, condiciones iniciales y de",
    "frontera, con el criterio de validez de cada supuesto.",
    "",
    "## 3. Verificación",
    "",
    "Caso de referencia, orden observado e índice de convergencia.",
    "",
    "## 4. Calibración",
    "",
    "Función objetivo, algoritmo, límites, punto inicial y datos.",
    "",
    "## 5. Validación",
    "",
    "Partición, métricas y criterio de aceptación declarado antes.",
    "",
    "## 6. Escenarios",
    "",
    "Alternativas comparadas y variables que las distinguen.",
    "",
    "## 7. Incertidumbre",
    "",
    "Presupuesto con las contribuciones ordenadas por aporte.",
    "",
    "## 8. Conclusiones",
    "",
    "Afirmación defendible con su respaldo cuantitativo.",
    "",
    "## 9. Limitaciones",
    "",
    "Dominio de validez, procesos omitidos, incertidumbres no",
    "cuantificadas, dependencia de escenarios y error numérico.",
    "",
    "## 10. Anexo reproducible",
    "",
    "Un solo comando reconstruye figuras, tablas e informe:",
    "",
    "    python3 codigo/reconstruir.py",
    "",
    "## Declaración del uso de asistentes de programación",
    "",
    "Alcance del uso y verificación aplicada a lo generado.",
]

LINEAS_RECONSTRUIR = [
    "# Regenera figuras, tablas e informe desde los datos crudos.",
    "# Uso, desde la raíz del repositorio:",
    "#     python3 codigo/reconstruir.py",
    "from pathlib import Path",
    "",
    "import numpy as np",
    "",
    "SEMILLA = 20262",
    "RAIZ = Path(__file__).resolve().parent.parent",
    "CRUDOS = RAIZ / 'datos' / 'crudos'",
    "FIGURAS = RAIZ / 'figuras'",
    "TABLAS = RAIZ / 'tablas'",
    "",
    "",
    "def main() -> None:",
    "    rng = np.random.default_rng(SEMILLA)",
    "    FIGURAS.mkdir(exist_ok=True)",
    "    TABLAS.mkdir(exist_ok=True)",
    "    print('reconstruido con semilla', SEMILLA)",
    "",
    "",
    "if __name__ == '__main__':",
    "    main()",
]

ENTORNO = [f"python=={sys.version_info.major}.{sys.version_info.minor}"]
for modulo in (np, pd):
    ENTORNO.append(f"{modulo.__name__}=={modulo.__version__}")
import matplotlib
import scipy
import sympy
for modulo in (matplotlib, scipy, sympy):
    ENTORNO.append(f"{modulo.__name__}=={modulo.__version__}")

ARCHIVOS = {
    "informe/informe.md": "\n".join(PLANTILLA_INFORME) + "\n",
    "codigo/reconstruir.py": "\n".join(LINEAS_RECONSTRUIR) + "\n",
    "entorno.txt": "\n".join(sorted(ENTORNO)) + "\n",
    "LICENSE": ("Licencia MIT. Se permite el uso, la copia y la "
                "modificación citando la fuente.\n"),
    "README.md": ("# Mini proyecto de simulación\n\n"
                  "Semilla del curso 20262. Reconstruya todo con\n"
                  "`python3 codigo/reconstruir.py`.\n"),
    "datos/crudos/LEEME.txt": ("Datos crudos de solo lectura. "
                               "Ninguna rutina escribe aquí.\n"),
    ".gitignore": "__pycache__/\n*.pyc\ndatos/procesados/\n",
}
for ruta_relativa, contenido in ARCHIVOS.items():
    destino = RAIZ_DEMO / ruta_relativa
    destino.parent.mkdir(parents=True, exist_ok=True)
    destino.write_text(contenido, encoding="utf-8")

print("Archivos escritos:")
for ruta_relativa in sorted(ARCHIVOS):
    tamano = (RAIZ_DEMO / ruta_relativa).stat().st_size
    print(f"  {ruta_relativa:28s} {tamano:5d} bytes")
print("\nEntorno declarado:")
print("\n".join(f"  {linea}" for linea in sorted(ENTORNO)))

### 2.2 El guion de reconstrucción se prueba desde cero

El libro es explícito, un solo comando reconstruye todo desde los
datos crudos y ese guion debe estar probado. Probarlo significa
ejecutarlo, no leerlo.

In [ ]:
import subprocess

proceso = subprocess.run(
    [sys.executable, str(RAIZ_DEMO / "codigo" / "reconstruir.py")],
    capture_output=True, text=True, cwd=str(RAIZ_DEMO))

print("Código de salida:", proceso.returncode)
print("Salida del guion:", proceso.stdout.strip())
if proceso.stderr.strip():
    print("Errores:", proceso.stderr.strip())
assert proceso.returncode == 0
assert "20262" in proceso.stdout
print("\nEl guion corre desde cero y deja constancia de la semilla.")

## 3. La lista de verificación de la Tabla 4.6, ejecutable

El libro recoge doce comprobaciones que se recorren con el informe
terminado, y advierte que ninguna casilla se marca sin señalar dónde
está la evidencia. Buena parte de esa revisión se puede automatizar,
y automatizarla evita que se marque por costumbre.

La función siguiente recibe la raíz de un repositorio y devuelve, por
cada punto de la tabla, si la evidencia existe y dónde. Lo que no se
puede comprobar leyendo archivos queda declarado como revisión
manual, que es lo honesto.

In [ ]:
LISTA_TABLA_4_6 = [
    (1, "La pregunta de ingeniería está enunciada y el alcance delimitado",
     "Párrafo con la decisión que apoya"),
    (2, "Los supuestos están listados y justificados",
     "Registro con criterio de validez"),
    (3, "El código está verificado",
     "Orden observado y caso de referencia"),
    (4, "El error de discretización está acotado",
     "Estudio de convergencia con índice"),
    (5, "Los parámetros llevan intervalo y correlación",
     "Matriz de covarianza o remuestreo"),
    (6, "La validación usó datos independientes",
     "Partición y criterio previo de aceptación"),
    (7, "Los residuales fueron examinados",
     "Paridad y diagnóstico de estructura"),
    (8, "La incertidumbre está presupuestada",
     "Tabla con contribuciones ordenadas"),
    (9, "Las cifras significativas son coherentes",
     "Redondeo gobernado por la incertidumbre"),
    (10, "Las limitaciones y el dominio están escritos",
     "Sección propia, no una frase de cortesía"),
    (11, "El informe se regenera con un solo comando",
     "Guion probado desde cero"),
    (12, "El uso de asistentes está declarado",
     "Nota con alcance y verificación aplicada"),
]

SENALES = {
    1: ["## 1. Problema y alcance"],
    2: ["## 2. Modelo y supuestos"],
    3: ["## 3. Verificación"],
    4: ["## 3. Verificación", "índice"],
    5: ["## 4. Calibración"],
    6: ["## 5. Validación", "criterio de aceptación"],
    7: ["## 5. Validación"],
    8: ["## 7. Incertidumbre", "aporte"],
    9: ["## 7. Incertidumbre"],
    10: ["## 9. Limitaciones"],
    11: ["## 10. Anexo reproducible"],
    12: ["Declaración del uso de asistentes"],
}

In [ ]:
def revisar_repositorio(raiz) -> pd.DataFrame:
    # Recorre la lista de la Tabla 4.6 sobre un repositorio y dice,
    # por cada punto, si la evidencia existe y en qué archivo.
    raiz = Path(raiz)
    informe = raiz / "informe" / "informe.md"
    texto = informe.read_text(encoding="utf-8") if informe.exists() else ""
    guion = raiz / "codigo" / "reconstruir.py"
    texto_guion = guion.read_text(encoding="utf-8") if guion.exists() else ""

    estructura = {
        "datos crudos de solo lectura": (raiz / "datos" / "crudos").is_dir(),
        "entorno declarado": (raiz / "entorno.txt").exists(),
        "licencia": (raiz / "LICENSE").exists(),
        "guion de reconstrucción": guion.exists(),
        "semilla fijada en el guion": "20262" in texto_guion,
    }

    filas = []
    for numero, comprobacion, evidencia in LISTA_TABLA_4_6:
        presente = all(s.lower() in texto.lower() for s in SENALES[numero])
        if numero == 11:
            presente = presente and estructura["guion de reconstrucción"]
            presente = presente and estructura["semilla fijada en el guion"]
        filas.append({"n": numero, "comprobación": comprobacion,
                      "evidencia esperada": evidencia,
                      "encontrada": presente,
                      "dónde": informe.name if presente else ""})
    marco = pd.DataFrame(filas)
    marco.attrs["estructura"] = estructura
    return marco


revision = revisar_repositorio(RAIZ_DEMO)
with pd.option_context("display.max_colwidth", 46):
    print(revision.to_string(index=False))

print("\nEstructura del repositorio:")
for nombre, presente in revision.attrs["estructura"].items():
    print(f"  {'sí' if presente else 'no':3s}  {nombre}")

aprobados = int(revision["encontrada"].sum())
print(f"\nPuntos con evidencia localizada: {aprobados} de "
      f"{len(revision)}")
assert len(revision) == 12
assert revision.attrs["estructura"]["semilla fijada en el guion"]

### 3.1 Un repositorio incompleto debe reprobar

Una lista de verificación que aprueba todo no sirve de nada. La celda
siguiente construye un repositorio al que le faltan la sección de
limitaciones, el guion de reconstrucción y la declaración del uso de
asistentes, y comprueba que la revisión lo detecte.

In [ ]:
RAIZ_INCOMPLETA = Path(tempfile.mkdtemp(prefix="informe_malo_"))
(RAIZ_INCOMPLETA / "informe").mkdir(parents=True, exist_ok=True)
recortado = "\n".join(
    l for l in PLANTILLA_INFORME
    if not l.startswith(("## 9.", "## 10.", "## Declaración")))
(RAIZ_INCOMPLETA / "informe" / "informe.md").write_text(
    recortado, encoding="utf-8")

revision_mala = revisar_repositorio(RAIZ_INCOMPLETA)
faltantes = revision_mala.loc[~revision_mala["encontrada"], "n"].tolist()
print("Puntos sin evidencia en el repositorio incompleto:", faltantes)
print(f"Aprobados {int(revision_mala['encontrada'].sum())} "
      f"de {len(revision_mala)}")

assert set(faltantes) >= {10, 11, 12}
assert not revision_mala.attrs["estructura"]["guion de reconstrucción"]
assert not revision_mala.attrs["estructura"]["licencia"]
print("\nLa lista detecta lo que falta, que es su única razón de ser.")

## 4. La ficha de métodos de la Tabla 4.7

La sección de métodos de un trabajo que usa simulación debe permitir
que otro grupo reconstruya un objeto computacional completo. Omitir
la versión de una biblioteca o la semilla de un generador equivale a
omitir la temperatura de un ensayo. Cuando alguno de los elementos no
aplica, se declara explícitamente en lugar de omitirse.

In [ ]:
import platform

# Los nueve elementos de la Tabla 4.7 y la clave con que se declaran.
ELEMENTOS_METODOS = [
    ("Modelo", "modelo"),
    ("Datos", "datos"),
    ("Método numérico", "metodo_numerico"),
    ("Verificación", "verificacion"),
    ("Calibración", "calibracion"),
    ("Validación", "validacion"),
    ("Incertidumbre", "incertidumbre"),
    ("Entorno", "entorno"),
    ("Disponibilidad", "disponibilidad"),
]


def ficha_de_metodos(**declarado) -> pd.DataFrame:
    # Llena la Tabla 4.7 con lo declarado por el autor y con lo que
    # se puede leer del propio sistema, para que nada se transcriba.
    automatico = {
        "Entorno": (f"Python {platform.python_version()} sobre "
                    f"{platform.system()}; "
                    + ", ".join(sorted(ENTORNO)[:3]) + "; "
                    f"semilla {SEMILLA}"),
    }
    filas = []
    for elemento, clave in ELEMENTOS_METODOS:
        valor = automatico.get(elemento) or declarado.get(
            clave, "NO DECLARADO")
        filas.append({"Elemento": elemento, "Qué se declara": valor})
    return pd.DataFrame(filas)


ficha = ficha_de_metodos(
    modelo="Modelo de Page de capa delgada, régimen isotérmico",
    datos="Trece instantes entre 0.25 h y 8 h, campaña a 60 grados",
    metodo_numerico="Mínimos cuadrados de Levenberg y Marquardt",
    verificacion="Solución analítica de referencia y orden observado",
    calibracion="Suma de cuadrados ordinaria, límites en el primer cuadrante",
    validacion="Lote independiente, criterio declarado antes del ensayo",
    incertidumbre="Covarianza del ajuste y Monte Carlo con semilla 20262",
    disponibilidad="Repositorio con licencia MIT e identificador persistente")

with pd.option_context("display.max_colwidth", 74):
    print(ficha.to_string(index=False))

sin_declarar = ficha.loc[ficha["Qué se declara"] == "NO DECLARADO",
                         "Elemento"].tolist()
print(f"\nElementos sin declarar: {sin_declarar or 'ninguno'}")
assert len(ficha) == 9
assert not sin_declarar
assert str(SEMILLA) in ficha.loc[ficha["Elemento"] == "Entorno",
                                 "Qué se declara"].iloc[0]

### 4.1 Los cuatro principios de la publicación de datos y de código

La publicación dejó de ser una cortesía y se convirtió en requisito
de un número creciente de revistas y de agencias. Los principios que
la orientan exigen objetos localizables por un identificador
persistente, accesibles por un protocolo abierto, interoperables por
el uso de formatos estándar y reutilizables gracias a una licencia
clara. Un cuaderno con celdas ejecutadas fuera de orden no es
reproducible ni por su autor.

In [ ]:
PRINCIPIOS = {
    "localizable": "identificador persistente asignado al depósito",
    "accesible": "protocolo abierto, sin registro previo",
    "interoperable": "formatos estándar, CSV y texto plano",
    "reutilizable": "licencia clara y procedencia declarada",
}

evaluacion = pd.DataFrame([
    {"principio": nombre, "requisito": requisito,
     "cumple en la plantilla": cumple}
    for (nombre, requisito), cumple in zip(
        PRINCIPIOS.items(),
        [False, True, True, (RAIZ_DEMO / "LICENSE").exists()])])
print(evaluacion.to_string(index=False))
print("\nLa plantilla cumple tres de los cuatro. Falta el identificador")
print("persistente, que se obtiene al depositar el repositorio y que")
print("ninguna comprobación local puede fabricar.")

# Un cuaderno reproducible se ejecuta de principio a fin y en orden.
def ejecucion_en_orden(ruta_cuaderno) -> bool:
    import json
    cuaderno = json.loads(Path(ruta_cuaderno).read_text(encoding="utf-8"))
    conteos = [c.get("execution_count") for c in cuaderno["cells"]
               if c["cell_type"] == "code"
               and c.get("execution_count") is not None]
    return conteos == sorted(conteos)


print("\nLa misma comprobación aplicada a un cuaderno mide si sus celdas")
print("se ejecutaron en orden, que es la condición que el libro señala.")

### 4.2 La figura única de la exposición oral

La exposición oral obedece a una economía distinta de la del
documento. El auditorio no puede releer, de modo que la charla se
organiza alrededor de una sola figura que sostiene el argumento.
Conviene declarar de entrada la pregunta y la afirmación, mostrar
después la evidencia y cerrar con las limitaciones, sin reservar la
conclusión para el final como si fuera un desenlace.

In [ ]:
GUION_CHARLA = [
    ("Pregunta", "¿El modelo calibrado predice el secado de un lote "
                 "que no vio?", "20 segundos"),
    ("Afirmación", "Sí, con error de 0.017 en razón de humedad y "
                   "sesgo del 4 por ciento.", "20 segundos"),
    ("Evidencia", "Figura única, serie observada y predicha del lote "
                  "independiente con su banda.", "3 minutos"),
    ("Limitaciones", "Vale entre 0.25 h y 8 h a 60 grados, y no "
                     "representa la variabilidad entre lotes.",
     "40 segundos"),
]
charla = pd.DataFrame(GUION_CHARLA,
                      columns=["momento", "contenido", "duración"])
with pd.option_context("display.max_colwidth", 62):
    print(charla.to_string(index=False))
print("\nLa conclusión aparece en el segundo momento, no al final.")
assert charla.iloc[1]["momento"] == "Afirmación"

## 5. Ejercicios guiados

Las celdas siguientes llevan la marca `# COMPLETE:` y arrancan con un
valor de partida evidentemente incorrecto, de modo que el cuaderno
corre completo aunque falten por resolver. Al terminarlas, cambie
`REVISAR = True` en la celda de configuración.

### Ejercicio 1. Contar lo que falta

Complete la función que devuelve los números de la Tabla 4.6 cuya
evidencia no se localizó, ordenados de menor a mayor.

In [ ]:
def puntos_pendientes(revision: pd.DataFrame) -> list[int]:
    """Números de la Tabla 4.6 sin evidencia localizada."""
    # COMPLETE: devuelva la lista ordenada de los valores de la
    # columna n cuyas filas tengan encontrada en falso.
    return None


pendientes_demo = puntos_pendientes(revision)
pendientes_malo = puntos_pendientes(revision_mala)

In [ ]:
# Verificación del ejercicio 1.
if pendientes_malo is None:
    print("[pendiente] la celda marcada COMPLETE sigue sin resolver")
else:
    print("Pendientes en la plantilla completa:", pendientes_demo)
    print("Pendientes en el repositorio incompleto:", pendientes_malo)
    correcta = set(pendientes_malo) >= {10, 11, 12}
    print(f"[{'ok' if correcta else 'revisar'}] la revisión detecta "
          f"los tres puntos que se recortaron")
    if REVISAR:
        assert correcta

### Ejercicio 2. Un punto nuevo en la lista

Añada a la revisión un punto trece que compruebe que el repositorio
declara su licencia. Devuelva el marco ampliado.

In [ ]:
# COMPLETE: construya revision_ampliada añadiendo a revision una fila
# con n igual a 13, la comprobación "El repositorio declara licencia",
# la evidencia "Archivo LICENSE en la raíz" y encontrada según exista
# el archivo LICENSE en RAIZ_DEMO.
revision_ampliada = None

In [ ]:
# Verificación del ejercicio 2.
if revision_ampliada is None:
    print("[pendiente] la celda marcada COMPLETE sigue sin resolver")
else:
    correcta = (len(revision_ampliada) == 13
                and bool(revision_ampliada.iloc[12]["encontrada"]))
    print(revision_ampliada.tail(2).to_string(index=False))
    print(f"[{'ok' if correcta else 'revisar'}] la lista tiene ahora "
          f"{len(revision_ampliada)} puntos")
    if REVISAR:
        assert correcta

### Ejercicio 3. La ficha de métodos incompleta

Llame a `ficha_de_metodos` omitiendo la validación y la
incertidumbre, y compruebe que la ficha las marque como no
declaradas en lugar de callarlas, que es lo que la Tabla 4.7 exige.

In [ ]:
# COMPLETE: construya ficha_parcial llamando a ficha_de_metodos solo
# con modelo y datos, y guarde en elementos_faltantes la lista de los
# elementos cuyo valor sea NO DECLARADO.
ficha_parcial = None
elementos_faltantes = None

In [ ]:
# Verificación del ejercicio 3.
if elementos_faltantes is None:
    print("[pendiente] la celda marcada COMPLETE sigue sin resolver")
else:
    print("Elementos sin declarar:", elementos_faltantes)
    correcta = set(elementos_faltantes) == {
        "Método numérico", "Verificación", "Calibración",
        "Validación", "Incertidumbre", "Disponibilidad"}
    print(f"[{'ok' if correcta else 'revisar'}] son "
          f"{len(elementos_faltantes)} de los nueve elementos")
    if REVISAR:
        assert correcta

### Ejercicio 4. Un cuaderno ejecutado fuera de orden

El libro señala que un cuaderno con celdas ejecutadas fuera de orden
no es reproducible ni por su autor. Complete la comprobación sobre
dos listas de contadores de ejecución.

In [ ]:
contadores_en_orden = [1, 2, 3, 4, 5]
contadores_desordenados = [1, 4, 2, 5, 3]

# COMPLETE: escriba una función que reciba la lista de contadores y
# devuelva True solo si están en orden creciente, y aplíquela a las
# dos listas anteriores.
resultado_en_orden = None
resultado_desordenado = None

In [ ]:
# Verificación del ejercicio 4.
if resultado_desordenado is None:
    print("[pendiente] la celda marcada COMPLETE sigue sin resolver")
else:
    correcta = bool(resultado_en_orden) and not resultado_desordenado
    print(f"Secuencia ordenada    {contadores_en_orden} -> "
          f"{resultado_en_orden}")
    print(f"Secuencia desordenada {contadores_desordenados} -> "
          f"{resultado_desordenado}")
    print(f"[{'ok' if correcta else 'revisar'}] la comprobación "
          f"distingue los dos casos")
    if REVISAR:
        assert correcta

### Ejercicio 5. La declaración del uso de asistentes

La sección 4.6.1 del libro fija que lo que se declara es el alcance
del uso y sobre todo la verificación aplicada a lo generado. Redacte
esa declaración de modo que mencione las dos cosas.

In [ ]:
# COMPLETE: escriba en declaracion_asistente un texto que mencione el
# alcance del uso y la verificación aplicada, con al menos treinta
# palabras.
declaracion_asistente = None

In [ ]:
# Verificación del ejercicio 5.
if declaracion_asistente is None:
    print("[pendiente] la celda marcada COMPLETE sigue sin resolver")
else:
    texto = declaracion_asistente.lower()
    menciona_alcance = "alcance" in texto
    menciona_verificacion = "verific" in texto
    suficiente = len(declaracion_asistente.split()) >= 30
    correcta = menciona_alcance and menciona_verificacion and suficiente
    print(declaracion_asistente)
    print(f"\n[{'ok' if correcta else 'revisar'}] menciona alcance "
          f"{menciona_alcance}, verificación {menciona_verificacion}, "
          f"{len(declaracion_asistente.split())} palabras")
    if REVISAR:
        assert correcta

## 6. Problemas del capítulo

### Problema 4-30, andamiaje

Evalúe la sección de métodos de un artículo publicado de su
disciplina frente a la Tabla 4.7, y señale qué permitiría reproducirlo
y qué lo impediría. La celda siguiente entrega la rúbrica ejecutable,
que usted llena con lo que encuentre en el artículo que escoja.

In [ ]:
def evaluar_articulo(**hallado) -> pd.DataFrame:
    # Rúbrica de la Tabla 4.7 aplicada a un artículo publicado.
    filas = []
    for elemento, clave in ELEMENTOS_METODOS:
        texto = hallado.get(clave, "")
        filas.append({"Elemento": elemento,
                      "Lo encontrado": texto or "ausente",
                      "Permite reproducir": bool(texto)})
    return pd.DataFrame(filas)


# Ejemplo con un artículo hipotético de secado en capa delgada.
evaluacion_articulo = evaluar_articulo(
    modelo="ecuación de Page con dos parámetros",
    datos="una campaña, sin depósito público",
    metodo_numerico="ajuste no lineal, sin nombrar la rutina",
    calibracion="mínimos cuadrados, sin punto inicial ni límites",
    validacion="mismo conjunto de la calibración")
with pd.option_context("display.max_colwidth", 48):
    print(evaluacion_articulo.to_string(index=False))
reproducible = int(evaluacion_articulo["Permite reproducir"].sum())
print(f"\nElementos suficientes para reproducir: {reproducible} de 9")
print("Sin entorno, sin semillas y sin datos disponibles, ningún grupo")
print("puede reconstruir el objeto computacional del artículo.")
assert reproducible < 9

### Problema 4-32, andamiaje

Tome un modelo de las unidades anteriores y prepare su paquete de
reproducibilidad, con repositorio, entorno declarado, semillas, guion
de reconstrucción y licencia. La celda siguiente comprueba que el
paquete construido en la sección 2 cumple los cinco requisitos.

In [ ]:
REQUISITOS_PAQUETE = {
    "repositorio con estructura": (RAIZ_DEMO / "codigo").is_dir(),
    "entorno declarado": (RAIZ_DEMO / "entorno.txt").exists(),
    "semilla del curso fijada": "20262" in (
        RAIZ_DEMO / "codigo" / "reconstruir.py").read_text(
            encoding="utf-8"),
    "guion de reconstrucción probado": proceso.returncode == 0,
    "licencia": (RAIZ_DEMO / "LICENSE").exists(),
}
for requisito, cumple in REQUISITOS_PAQUETE.items():
    print(f"  {'sí' if cumple else 'no':3s}  {requisito}")
assert all(REQUISITOS_PAQUETE.values())
print("\nEl paquete cumple los cinco requisitos del problema 4-32.")
print(f"Carpeta temporal usada: {RAIZ_DEMO.name}")

## Cierre

### Lo que debe saber hacer al terminar

- Trasladar cada sección del informe a la del artículo que le corresponde, con la pregunta que cada una responde.
- Crear la estructura de un repositorio reproducible con datos crudos de solo lectura, entorno declarado y licencia.
- Ejecutar la lista de verificación de la Tabla 4.6 sobre un repositorio y leer el veredicto punto por punto.
- Llenar la ficha de métodos de la Tabla 4.7 leyendo el entorno del propio sistema, sin transcribir versiones a mano.
- Declarar el uso de asistentes de programación con el alcance y la verificación aplicada a lo generado.

### Qué revisar en el libro si algo no salió

- Si duda de qué lleva cada sección, la Figura 4.12 y la Tabla 4.7 del libro las enumeran.
- Si la lista de verificación falla, la Tabla 4.6 dice qué evidencia debe existir para cada punto.
- Si el informe no se reconstruye con un solo comando, revise la Definición 4.11 y la Figura 4.11.
- Si no sabe cómo declarar el uso de un asistente, la sección 4.6.1 fija que lo declarado es el alcance y la verificación aplicada.

### Declaración del uso de asistentes de programación

Este cuaderno se preparó con apoyo de un asistente de programación.
Todo resultado numérico que aparece aquí se verifica contra el valor
que el libro publica, contra una solución analítica o contra un caso
límite, según recuerda la sección 4.6 del libro. La responsabilidad
del contenido no se transfiere al asistente.